# Structural preprocessing

## Notebook preparation

In [1]:
import os
import os.path as op

from utils import loadFSL, FSLeyesServer, mkdir_no_exist

# Load FSL within Neurodesk
import module as lmod

await lmod.purge(force=True)
await lmod.load("fsl/6.0.7.18")

loadFSL()

# Image loading and manipulation
import numpy as np
import nibabel as nib

# Structural preprocessing
from fsl.wrappers import bet, fast, flirt, fslmaths

# Advanced option: nonlinear registration
import ants

# Visualization and report figures
import matplotlib.pyplot as plt
from nilearn import plotting

In [2]:
%gui wx


(ipykernel_launcher.py:2162): dbind-WARNING **: 14:35:12.286: Couldn't connect to accessibility bus: Failed to connect to socket /run/user/1416591259/at-spi/bus_100: No such file or directory


In [3]:
# Start FSLeyes
fsleyesDisplay = FSLeyesServer()
fsleyesDisplay.show()

14:35:16: Debug: Adding duplicate image handler for 'Windows bitmap file'
14:35:16: Debug: Adding duplicate animation handler for '1' type
14:35:16: Debug: Adding duplicate animation handler for '2' type
14:35:16: Debug: Adding duplicate image handler for 'Windows bitmap file'
14:35:16: Debug: Adding duplicate animation handler for '1' type
14:35:16: Debug: Adding duplicate animation handler for '2' type

(ipykernel_launcher.py:2162): Gtk-CRITICAL **: 14:35:16.388: gtk_window_resize: assertion 'width > 0' failed

(ipykernel_launcher.py:2162): Gtk-CRITICAL **: 14:35:18.321: gtk_window_resize: assertion 'width > 0' failed


In [5]:
# Dataset organization
dataset_root = op.abspath("dataset")
anatomical_path = op.join(dataset_root, "T1", "T1w.nii.gz")

# Save preprocessing outputs inside dataset/derivatives/anat
preproc_root = op.join(dataset_root, "derivatives")
anat_preproc_dir = op.join(preproc_root, "anat")
os.makedirs(anat_preproc_dir, exist_ok=True)

## Skull stripping

In [6]:
# Set BET result path for betted image and mask
betted_brain_path = op.join(anat_preproc_dir, "T1w_betted.nii.gz")
resulting_mask_path = op.join(anat_preproc_dir, "T1w_mask.nii.gz")

# Perform BET
# when setting to True manual correction should then be done
PERFORM_BET = False
if PERFORM_BET:
    os.system('bet "{}" "{}" -m -R'.format(anatomical_path, betted_brain_path))
    print("BET skull stripping done")

In [7]:
# Check the resulting mask to apply manual correction
fsleyesDisplay.resetOverlays()
fsleyesDisplay.load(anatomical_path)
fsleyesDisplay.load(resulting_mask_path)

In [8]:
# Apply mask with FSL method
os.system('fslmaths {} -mas {} {}'.format(anatomical_path, resulting_mask_path, betted_brain_path))

0

In [9]:
# Check skull stripped brain
fsleyesDisplay.resetOverlays()

fsleyesDisplay.load(anatomical_path)
fsleyesDisplay.load(betted_brain_path)

## Segmentation

In [10]:
# Create path for segmented image
segmentation_path = op.join(anat_preproc_dir, "T1w_fast")

# Perform FAST segmentation
PERFORM_FAST = False
if PERFORM_FAST:
    fast(imgs=[betted_brain_path], out=segmentation_path, n_classes=3)

In [11]:
# Visualize segmentation result
fsleyesDisplay.resetOverlays()

fsleyesDisplay.load(betted_brain_path)
fsleyesDisplay.load(segmentation_path + "_pve_0.nii.gz")
fsleyesDisplay.load(segmentation_path + "_pve_1.nii.gz")
fsleyesDisplay.load(segmentation_path + "_pve_2.nii.gz")

fsleyesDisplay.displayCtx.getOpts(fsleyesDisplay.overlayList[1]).cmap = 'Red'
fsleyesDisplay.displayCtx.getOpts(fsleyesDisplay.overlayList[2]).cmap = 'Green'
fsleyesDisplay.displayCtx.getOpts(fsleyesDisplay.overlayList[3]).cmap = 'Blue'

## Non-linear Registration

In [12]:
# Set skull-stripped MNI template as reference image
reference_mni = op.expandvars("$FSLDIR/data/standard/MNI152_T1_1mm_brain.nii.gz")

moving_image = ants.image_read(betted_brain_path)
fixed_image = ants.image_read(reference_mni)

# Compute the non linear transformation to put align the moving image to the fixed image
transformation = ants.registration(fixed=fixed_image,
                                   moving=moving_image,
                                   type_of_transform = 'SyN')

# Apply the computed transformation
warpedImage = ants.apply_transforms(fixed=fixed_image,
                                    moving=moving_image,
                                    transformlist=transformation["fwdtransforms"])

# Save the registered brain
registration_path = op.join(anat_preproc_dir, "T1w_mni.nii.gz")
ants.image_write(warpedImage, registration_path)

In [17]:
# Check source image and MNI template before registration
fsleyesDisplay.resetOverlays()

fsleyesDisplay.load(reference_mni)
fsleyesDisplay.load(betted_brain_path)

# Source image with specific color map
overlay = fsleyesDisplay.overlayList[1] 
opts = fsleyesDisplay.displayCtx.getOpts(overlay)

opts.cmap = "hot"
fsleyesDisplay.displayCtx.getDisplay(overlay).alpha = 50

In [18]:
# Check source image and MNI template after registration
fsleyesDisplay.resetOverlays()

fsleyesDisplay.load(reference_mni)
fsleyesDisplay.load(registration_path)

# Source image with specific color map
overlay = fsleyesDisplay.overlayList[1] 
opts = fsleyesDisplay.displayCtx.getOpts(overlay)

opts.cmap = "hot"
fsleyesDisplay.displayCtx.getDisplay(overlay).alpha = 50

<Figure size 640x480 with 0 Axes>